In [5]:
import re
import pandas as pd
import glob
import os

def procesar_log_detallado(ruta_archivo):
    try:
        with open(ruta_archivo, 'r', encoding='utf-8') as f:
            texto = f.read()
    except FileNotFoundError:
        return f"Error: No se encontró el archivo {ruta_archivo}", None, None

    # Separamos el log por secciones asociadas a cada dataset
    secciones = re.split(r"\*{20,}Ejecutando prueba sobre el dataset", texto)
    
    resumen_data = []
    clases_data = []
    
    # Extraemos el nombre del método de la primera sección o del log en general si es posible
    # Buscamos la línea: Configuración: Epoch=..., Aumentado=Nombre del Método
    metodo_match_global = re.search(r"Aumentado=(.+)$", texto, re.MULTILINE)

    if metodo_match_global:
        nombre_metodo_defecto = metodo_match_global.group(1).strip().replace("Aumentado=", "")
    else:
        nombre_metodo_defecto = os.path.basename(ruta_archivo).replace(".log", "").replace("Aumentado=", "")

    for seccion in secciones[1:]:
        lineas = seccion.strip().split('\n')
        dataset_name = lineas[0].split('*')[0].strip()
        
        # Extraemos métricas
        oa_match = re.search(r"OA Final:\s*([\d.]+)%\s*±\s*([\d.]+)%", seccion)
        aa_match = re.search(r"AA Final:\s*([\d.]+)%\s*±\s*([\d.]+)%", seccion)
        tiempo_total_match = re.search(r"Tiempo entrenamiento total medio .*?:\s*([\d.]+)\s*s", seccion)
        tiempo_epoca_match = re.search(r"Tiempo medio por época .*?:\s*([\d.]+)\s*s", seccion)
        
        # Extraemos el método específico de esta sección (por si acaso varía, aunque suele ser el mismo por log)
        metodo_match = re.search(r"Aumentado=(.+)$", seccion, re.MULTILINE)
        nombre_metodo = metodo_match.group(1).strip().replace("Aumentado=", "") if metodo_match else nombre_metodo_defecto

        if oa_match and aa_match:
            datos_dataset = {
                "Método": nombre_metodo,
                "Dataset": dataset_name,
                "OA (%)": float(oa_match.group(1)),
                "std OA (%)": float(oa_match.group(2)),
                "AA (%)": float(aa_match.group(1)),
                "std AA (%)": float(aa_match.group(2)),
                "Tiempo Total (s)": float(tiempo_total_match.group(1)) if tiempo_total_match else None,
                "Tiempo por Época (s)": float(tiempo_epoca_match.group(1)) if tiempo_epoca_match else None
            }
            resumen_data.append(datos_dataset)
            
        bloque_clases = re.search(r"ACCURACY POR CLASE:(.*?)---", seccion, re.DOTALL)
        if bloque_clases:
            matches_clase = re.findall(r"Clase (\d+):\s*([\d.]+)%", bloque_clases.group(1))
            for num_clase, valor_acc in matches_clase:
                clases_data.append({
                    "Dataset": dataset_name,
                    "Clase": f"Clase {num_clase}",
                    "Accuracy (%)": float(valor_acc)
                })
    
    if not resumen_data:
        return "No se encontraron datos válidos.", None, None

    df_resumen = pd.DataFrame(resumen_data)
    df_clases_long = pd.DataFrame(clases_data)
    df_clases_wide = df_clases_long.pivot(index='Dataset', columns='Clase', values='Accuracy (%)')
    
    stats = {
        "Métrica": ["OA Global", "AA Global", "Tiempo Total (s)", "Tiempo por Época (s)"],
        "Media": [df_resumen["OA (%)"].mean(), df_resumen["AA (%)"].mean(), df_resumen["Tiempo Total (s)"].mean(), df_resumen["Tiempo por Época (s)"].mean()],
        "Desviación": [df_resumen["OA (%)"].std(), df_resumen["AA (%)"].std(), df_resumen["Tiempo Total (s)"].std(), df_resumen["Tiempo por Época (s)"].std()]
    }
    df_stats = pd.DataFrame(stats)
    
    # Retornamos también el nombre del método para usarlo en las gráficas
    return df_resumen, df_stats, df_clases_wide, nombre_metodo_defecto

In [6]:
def imprimirResultados(ruta_carpeta):
    archivos = glob.glob(os.path.join(ruta_carpeta, "*.log"))
    datos_comparacion = []

    if not archivos:
        print(f"No se encontraron archivos .log en la carpeta '{ruta_carpeta}'.")
        return []

    for archivo in archivos:
        nombre_archivo = os.path.basename(archivo).replace(".log", "")
        print(f"\n{'='*30} ARCHIVO: {nombre_archivo} {'='*30}")
        
        df_res, df_st, df_clases, nombre_metodo = procesar_log_detallado(archivo)
        
        if isinstance(df_res, pd.DataFrame):
            print("\n************* TABLA RESUMEN (OA/AA): *************")
            print(df_res.to_string(index=False))
            print("\n************* ESTADÍSTICAS GLOBALES: *************")
            print(df_st.to_string(index=False))
            
            oa_global = df_st.loc[df_st['Métrica'] == 'OA Global', 'Media'].values[0]
            aa_global = df_st.loc[df_st['Métrica'] == 'AA Global', 'Media'].values[0]
            oa_std = df_st.loc[df_st['Métrica'] == 'OA Global', 'Desviación'].values[0]
            aa_std = df_st.loc[df_st['Métrica'] == 'AA Global', 'Desviación'].values[0]
            
            t_total_medio = df_st.loc[df_st['Métrica'] == 'Tiempo Total (s)', 'Media'].values[0]
            t_epoca_medio = df_st.loc[df_st['Métrica'] == 'Tiempo por Época (s)', 'Media'].values[0]
            
            datos_comparacion.append({
                "Método": nombre_metodo,  # Usamos el nombre descriptivo extraído
                "OA Global (%)": oa_global,
                "AA Global (%)": aa_global,
                "OA Std (%)": oa_std,
                "AA Std (%)": aa_std,
                "Tiempo Total Medio (s)": t_total_medio,
                "Tiempo Época Medio (s)": t_epoca_medio
            })
        else:
            print(df_res)
        
        if datos_comparacion:
            datos_ordenados = sorted(
                datos_comparacion, 
                key=lambda x: (x["AA Global (%)"], x["OA Global (%)"]), 
                reverse=True
            )
            
            # Nos quedamos con los 5 primeros (o menos si no hay 5 archivos)
            top_5 = datos_ordenados[:5]
            
            print(f"\n\n{'='*25} TOP 5 MEJORES RESULTADOS {'='*25}")
            print(f"{'Ranking':<8} | {'Método':<30} | {'OA Global (%) ± Std':<22} | {'AA Global (%) ± Std':<22}")
            print("-" * 91)
            
            for i, dato in enumerate(top_5, start=1):
                # Formateamos el texto para incluir la media y la desviación estándar
                oa_str = f"{dato['OA Global (%)']:.2f} ± {dato['OA Std (%)']:.2f}"
                aa_str = f"{dato['AA Global (%)']:.2f} ± {dato['AA Std (%)']:.2f}"
                
                print(f"#{i:<7} | {dato['Método']:<30} | {oa_str:<22} | {aa_str:<22}")
                
            print("=" * 91)
    return datos_comparacion

In [7]:
import matplotlib.pyplot as plt

def graficaComparacionModelos(datos_comparacion):
    if datos_comparacion:
        df_comparacion = pd.DataFrame(datos_comparacion)

        if 'AA Global (%)' in df_comparacion.columns:
            df_comparacion = df_comparacion.sort_values(by='AA Global (%)', ascending=False)
        
        tiene_tiempos = "Tiempo Total Medio (s)" in df_comparacion.columns and not df_comparacion["Tiempo Total Medio (s)"].isnull().all()
        
        num_graficas = 3 if tiene_tiempos else 2
        # Hacemos la figura un poco más ancha para alojar cómodamente más etiquetas
        fig, axes = plt.subplots(num_graficas, 1, figsize=(14, 6 * num_graficas), sharex=False)
        
        ax1 = axes[0]
        ax2 = axes[1]

        # Gráfica Precisión
        df_comparacion.set_index('Método')[['OA Global (%)', 'AA Global (%)']].plot(
            kind='bar', ax=ax1, color=['#1f77b4', '#ff7f0e'], alpha=0.8
        )
        ax1.set_title('Media Precisión Global (OA y AA)', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Precisión (%)')
        ax1.set_ylim(80, 100)
        ax1.grid(axis='y', linestyle='--', alpha=0.5)
        ax1.legend(["OA Global", "AA Global"], loc='upper right')

        # Gráfica Desviación
        df_comparacion.set_index('Método')[['OA Std (%)', 'AA Std (%)']].plot(
            kind='bar', ax=ax2, color=['#2ca02c', '#d62728'], alpha=0.8
        )
        ax2.set_title('Desviación estándar (OA y AA)', fontsize=14, fontweight='bold')
        ax2.set_ylabel('Variación (%)')
        ax2.grid(axis='y', linestyle='--', alpha=0.5)
        ax2.legend(["OA Std", "AA Std"], loc='upper right')

        # Gráfica Tiempos
        if tiene_tiempos:
            ax3 = axes[2]
            df_comparacion.set_index('Método')[['Tiempo Época Medio (s)']].plot(
                kind='bar', ax=ax3, color=['#8c564b'], alpha=0.8
            )
            ax3.set_title('Coste Computacional (Tiempo Medio por Época)', fontsize=14, fontweight='bold')
            ax3.set_ylabel('Tiempo (segundos)')
            ax3.set_ylim(bottom=0)
            ax3.grid(axis='y', linestyle='--', alpha=0.5)
            ax3.legend(["Tiempo Medio por Época"], loc='upper right')

        plt.xticks(rotation=45, ha='right')
        plt.xlabel('Métodos de Aumentado')
        plt.tight_layout()
        plt.show()

## Resultados usando el aumentado de la clase minoritaria en el dataloader

In [8]:
ruta_carpeta = "Codigos_AumentadosPyTorch/resultadosAumentado_PyTorch"

datos_comparacion=imprimirResultados(ruta_carpeta)


============================== ARCHIVO: metodo_0_Sin_Aumentado ==============================


ValueError: not enough values to unpack (expected 4, got 3)

In [9]:
graficaComparacionModelos(datos_comparacion)

NameError: name 'datos_comparacion' is not defined

# Comportamiento de las distintas versiones por cada dataset

In [10]:
def graficar_clases_por_dataset(dataset_objetivo, ruta_carpeta="resultados_logs"):
    archivos = glob.glob(os.path.join(ruta_carpeta, "*.log"))
    df_clases_metodos = pd.DataFrame()

    if not archivos:
        print(f"No hay logs en la carpeta {ruta_carpeta}")
        return

    # Recolectar datos
    for archivo in archivos:
        df_res, _, df_clases, nombre_metodo = procesar_log_detallado(archivo)
        
        # Extraemos la fila de accuracies para este dataset si existe en el log
        if isinstance(df_clases, pd.DataFrame) and dataset_objetivo in df_clases.index:
            df_clases_metodos[nombre_metodo] = df_clases.loc[dataset_objetivo]

    if df_clases_metodos.empty:
        print(f"No hay datos para el dataset: {dataset_objetivo}")
        return

    # Limpiar clases vacías
    df_clases_metodos = df_clases_metodos.dropna(how='all')
    df_clases_metodos = df_clases_metodos[(df_clases_metodos > 0).any(axis=1)]

    # Determinar el ymin dinámico
    valores_validos = df_clases_metodos.values.flatten()
    valores_validos = [v for v in valores_validos if v > 0 and not pd.isna(v)]
    ymin = (min(valores_validos) // 10) * 10 if valores_validos else 0

    # Dibujar la gráfica
    fig, ax = plt.subplots(figsize=(15, 7))
    
    # tab20 nos da suficientes colores distintos si comparas hasta 20 métodos
    df_clases_metodos.plot(kind='bar', ax=ax, width=0.85, colormap='tab20')
    
    ax.set_title(f'Dataset: {dataset_objetivo} - Precisión por clase según Método de Aumentado', fontsize=15, fontweight='bold')
    ax.set_ylim(ymin, 100)
    ax.set_ylabel('Accuracy (%)')
    ax.set_xlabel('Clases')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    
    # Movemos la leyenda fuera para que no tape las barras si hay 11 métodos
    ax.legend(title="Métodos de Aumentado", loc='center left', bbox_to_anchor=(1, 0.5))

    plt.tight_layout()
    plt.show()

# Ejecución general
def ejecutar_graficas_dinamicas(ruta_carpeta="resultados_logs"):
    archivos = glob.glob(os.path.join(ruta_carpeta, "*.log"))
    if not archivos:
        print("No se detectaron datasets.")
        return
        
    _, _, df_clases_ejemplo, _ = procesar_log_detallado(archivos[0])
    
    if isinstance(df_clases_ejemplo, pd.DataFrame):
        datasets_disponibles = df_clases_ejemplo.index.tolist()
        print(f"Datasets detectados: {datasets_disponibles}")
        for ds in datasets_disponibles:
            print(f"\nGenerando gráfica para: {ds}...")
            graficar_clases_por_dataset(ds, ruta_carpeta)

In [11]:
ejecutar_graficas_dinamicas()

No se detectaron datasets.
